# GAT Session Recommendation Results Presentation

This notebook presents the GAT-based session recommendation experiments as a results story:

1. Compare the **best achieved trained results** against the matching published or contextual baseline.
2. Explain how each direct model evolved across result iterations.
3. Present **GAT-SAGPool** as the third architecture, separately from the controlled SR-GNN/TAGNN readout comparisons.
4. Keep missing optional artifacts explicit, especially the separate `GAT-SAGPool` folder when it is not available.


## Executive Summary

- **Iteration 9 is trained for both direct branches**. It keeps the restored weighted `GATConv(edge_dim=1)` architecture from iteration 8, restores dropout `0.10`, and adds small label smoothing `0.01` through `r5_restored_baseline_label_smoothing`.
- **Label smoothing helped Yoochoose most clearly**. `GAT-SR-GNN` reaches a new best Yoochoose MRR@20 of `31.172`, and `GAT-TAGNN` reaches new best Yoochoose Precision@20 / MRR@20 of `70.645` / `31.038`.
- **Diginetica is mixed**. `GAT-TAGNN` iteration 9 sets the best direct TAGNN-style GAT Diginetica Precision@20 at `51.351`, but its MRR@20 `17.345` is slightly below iteration 8 `17.374`; `GAT-SR-GNN` Diginetica remains best in `results-3`.
- **The strongest overall direct GAT checkpoints are now split across iterations**: SR-GNN Yoochoose MRR from `results-9`, SR-GNN Diginetica from `results-3`, TAGNN Yoochoose from `results-9`, and TAGNN Diginetica MRR from `results-8`.
- **Practical conclusion**: keep label smoothing `0.01` for the next Yoochoose-focused run, but do not increase it before confirming Diginetica MRR because the ranking-quality gain there was not consistent.


## Result Iterations

| Iteration | Folder | GAT-SR-GNN status | GAT-TAGNN status | What Changed |
| --- | --- | --- | --- | --- |
| 1 | results-1 | trained | trained | Initial complete GAT-SR-GNN and GAT-TAGNN archive used as the first working reference point. |
| 2 | results-2 | trained | pending | Simplified GAT stack revision; only GAT-SR-GNN was rerun in this export. |
| 3 | results-3 | trained | trained | GATv2/performance iteration, including TAGNN scoring/chunking changes and SR-GNN positional-embedding rollback. |
| 4 | results-4 | trained | pending | Repeated transition edges plus explicit residual connections in the GAT stack; TAGNN still pending. |
| 5 | results-5 | trained | trained | Failed edge-attribute/gated-normalization ablation: normalized `edge_attr` for `GATv2Conv(edge_dim=1)` with gated residual blending and `LayerNorm` in each directional GAT stack. |
| 6 | results-6 | trained | trained | Shared GAT-only recovery iteration: repeated transition edges by default, no scalar `edge_attr`, `GATv2Conv`, plain residual, one layer; SR-GNN/TAGNN readouts unchanged. |
| 7 | results-7 | trained | trained | GAT-only identity-preserving fusion: same iteration-6 GATv2 stack, then add original item embedding after direction fusion; readouts unchanged. |
| 8 | results-8 | trained | trained | Restored iteration-1 weighted `GATConv(edge_dim=1)` architecture from commit `7bff12d`; first regularization test uses mild dropout `0.15` with no LayerNorm/gating/identity skip. |
| 9 | results-9 | trained | trained | Same weighted `GATConv(edge_dim=1)` architecture as iteration 8, but uses dropout `0.10` plus small label smoothing `0.01` via `r5_restored_baseline_label_smoothing`. |

Iteration 9 changes only the regularization preset, so the SR-GNN and TAGNN second phases remain controlled against their paper-style definitions. The run suggests very small label smoothing is useful for Yoochoose and Precision@20, while Diginetica MRR remains sensitive.


## Aktualne Schematy Architektur

Wspólne wejście grafowe: każdy prefiks sesji jest zamieniany na skierowany graf przejść między itemami, z osobnymi indeksami krawędzi forward i backward. Aktualny kod treningowy odpowiada przetestowanej iteracji 9: oba modele utrzymują ważone krawędzie przejść i `GATConv(edge_dim=1)` z iteracji 8, ale wracają do historycznie używanego dropout `0.10`. Powtarzające się przejścia są agregowane, a nie przechowywane jako powtórzone krawędzie.

**Wspólny GAT encoder dla GAT-SR-GNN i GAT-TAGNN**

```text
Graf prefiksu sesji
  -> itemy unikalne jako node'y
  -> krawędzie forward/backward z kolejnych kliknięć
  -> duplicate transitions agregowane do edge weights
  -> forward weight: count / out_degree(source)
  -> backward weight: count / in_degree(target)
  -> item embedding: item_id -> 100
  -> bidirectional weighted GAT encoder
       forward stack: GATConv(edge_dim=1), 1 layer, 4 heads x 25 dims -> 100
       backward stack: GATConv(edge_dim=1), 1 layer, 4 heads x 25 dims -> 100
       update: residual + dropout(ELU(GATConv(..., edge_attr)))
       dropout default: 0.10, label smoothing: 0.01 for r5_restored_baseline_label_smoothing
       brak gated residual blending i brak LayerNorm
  -> direction fusion: concat(100 + 100) -> linear 200 -> 100
  -> brak post-fusion identity skip
  -> contextual node embeddings
```

**GAT-SR-GNN readout (bez zmian w iteracjach 8-9)**

```text
contextual node embeddings
  -> mapowanie node embeddings z powrotem do kolejności prefiksu
  -> local preference: embedding ostatniego kliknięcia
  -> global preference: SR-GNN attention po node'ach prefiksu warunkowana ostatnim kliknięciem
  -> hybrid projection: concat(local 100 + global 100) -> 100
  -> wyniki dot-product względem embeddingów wszystkich itemów
```

**GAT-TAGNN readout (bez zmian w iteracjach 8-9)**

```text
contextual node embeddings
  -> padding node embeddings w kolejności prefiksu + sequence mask
  -> base session preference: global attention warunkowana ostatnim kliknięciem
  -> target-aware candidate scoring TAGNN
       candidate chunks zgodnie z configiem
       target attention: candidate embedding x przetransformowane node'y prefiksu
       final score: base dot-product + target-aware score
```


## Architecture Evolution Logic (Iterations 1 -> Current Code)

| Iteration | Main change | Why | Effect on architecture/results |
| --- | --- | --- | --- |
| 1 | Baseline GAT notebooks with architecture docs (`GATConv`, weighted edges, explicit residual) | First complete working GAT replacement for SR-GNN/TAGNN pipelines | Established the strongest early TAGNN Yoochoose result and best SR-GNN Yoochoose MRR until iteration 9. |
| 2 | Simplification pass: removed edge-weight path and manual residual in stack; kept native attention/self-loops | Reduce optimization friction and let attention learn edge importance directly | SR-GNN Diginetica improved over iteration 1, but Yoochoose dropped. |
| 3 | Performance pass: moved to `GATv2Conv`; SR removed positional embedding path; TAGNN scoring/chunking updated | Improve attention expressiveness and scoring efficiency | Best SR-GNN Diginetica result; TAGNN did not improve over iteration 1. |
| 4 | Reintroduced explicit residual update and kept repeated transitions in graph construction (`0a15170`) | Preserve transition frequency signal and stabilize feature updates | SR-GNN stayed near iteration 3 on Diginetica but did not set a new best. |
| 5 | Aggregated duplicate transitions as normalized `edge_attr` for `GATv2Conv(edge_dim=1)` and added gated residual `LayerNorm` | Test explicit transition-frequency bias and stronger update stabilization | Regressed sharply on both datasets/readouts; treated as a failed ablation. |
| 6 | Shared GAT-only cleanup: repeated edges by default, no scalar edge features, plain residual, no LayerNorm, one shared GAT config for both models | Recover from iteration-5 regression while keeping SR-GNN/TAGNN second phases unchanged and synchronized | Partial recovery, especially SR-GNN and TAGNN Diginetica, but no new historical best. |
| 7 | Post-fusion identity skip: return `item_embedding + direction_fusion([forward, backward])` | Preserve raw item identity for the unchanged readouts while keeping GAT context additive | Trained for both direct branches, but regressed versus iteration 6 on both datasets/readouts. |
| 8 | Roll back to iteration-1 weighted `GATConv(edge_dim=1)` and apply mild dropout `0.15` | Recover the strongest historical architecture while testing one regularization increase | New best TAGNN-style GAT MRR on Diginetica; no new best for SR-GNN or Yoochoose TAGNN by MRR. |
| 9 | Keep iteration-8 weighted `GATConv(edge_dim=1)`, restore dropout `0.10`, and add label smoothing `0.01` | Test a small confidence regularizer without changing encoder/readout math | New best Yoochoose MRR for SR-GNN-style GAT, new best Yoochoose TAGNN-style GAT, and new best TAGNN-style GAT Diginetica Precision@20; Diginetica MRR is mixed. |

Cause -> effect chain: iteration 8 confirmed that the restored weighted `GATConv` design is still competitive, especially for TAGNN on Diginetica. Iteration 9 keeps the architecture fixed, changes dropout from `0.15` back to `0.10`, and adds label smoothing `0.01`; this reduced overconfidence enough to help Yoochoose ranking metrics, but it did not improve Diginetica MRR consistently.


## Best Achieved Architecture Results vs Baselines

The main comparison selects the best trained run for each direct architecture and dataset by `MRR@20`, using `Precision@20` as a tie-breaker. `GAT-SR-GNN` is compared with published **SR-GNN**, and `GAT-TAGNN` with published **TAGNN**. `GAT-SAGPool` is omitted from this snapshot because `results-gat-sagpool/` is not present in the current workspace.

| Architecture | Dataset | Best Run | Baseline | Precision@20 | Baseline P@20 | Delta P@20 | MRR@20 | Baseline MRR@20 | Delta MRR@20 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| GAT-SR-GNN | Diginetica | results-3 | SR-GNN | 50.736 | 50.730 | +0.006 | 16.855 | 17.590 | -0.735 |
| GAT-SR-GNN | Yoochoose 1/64 | results-9 | SR-GNN | 70.174 | 70.570 | -0.396 | 31.172 | 30.940 | +0.232 |
| GAT-TAGNN | Diginetica | results-8 | TAGNN | 51.313 | 51.310 | +0.003 | 17.374 | 18.030 | -0.656 |
| GAT-TAGNN | Yoochoose 1/64 | results-9 | TAGNN | 70.645 | 71.020 | -0.375 | 31.038 | 31.120 | -0.082 |

Iteration 9 replaces the previous best Yoochoose rows for both direct GAT branches by MRR@20: `GAT-SR-GNN` improves Yoochoose MRR from `31.146` to `31.172`, and `GAT-TAGNN` improves Yoochoose MRR from `31.004` to `31.038`. Diginetica remains split: `GAT-SR-GNN` is still strongest in `results-3`, while `GAT-TAGNN` keeps its best Diginetica MRR in `results-8` even though iteration 9 has the highest Diginetica Precision@20.


![Best GAT architecture results compared with baselines](results-analysis/best_vs_baseline.png)

![Best-result deltas from the matching baseline](results-analysis/best_delta_vs_baseline.png)

![Best direct comparison across GAT architectures](results-analysis/architecture_best_comparison.png)


## Model Evolution: GAT-SR-GNN

`GAT-SR-GNN` now has a complete nine-iteration trained trace. Iteration 9 keeps the weighted `GATConv(edge_dim=1)` encoder from iteration 8, restores dropout `0.10`, and adds label smoothing `0.01`. This becomes the best SR-GNN-style GAT Yoochoose run by MRR@20 (`31.172`), while Diginetica remains best in `results-3`.


| Dataset | Run | Status | Precision@20 | MRR@20 | Best Epoch | Best Val MRR@20 |
| --- | --- | --- | --- | --- | --- | --- |
| Diginetica | results-1 | trained | 50.250 | 16.646 | 4 | 19.413 |
| Diginetica | results-2 | trained | 50.629 | 16.725 | 28 | 19.486 |
| Diginetica | results-3 | trained | 50.736 | 16.855 | 19 | 19.563 |
| Diginetica | results-4 | trained | 50.454 | 16.802 | 19 | 19.529 |
| Diginetica | results-5 | trained | 48.173 | 15.789 | 10 | 18.889 |
| Diginetica | results-6 | trained | 50.391 | 16.759 | 28 | 19.532 |
| Diginetica | results-7 | trained | 50.266 | 16.696 | 26 | 19.471 |
| Diginetica | results-8 | trained | 50.299 | 16.569 | 4 | 19.462 |
| Diginetica | results-9 | trained | 50.332 | 16.664 | 4 | 19.447 |
| Yoochoose 1/64 | results-1 | trained | 70.124 | 31.146 | 9 | 32.706 |
| Yoochoose 1/64 | results-2 | trained | 68.729 | 30.355 | 29 | 31.680 |
| Yoochoose 1/64 | results-3 | trained | 68.942 | 30.563 | 25 | 31.759 |
| Yoochoose 1/64 | results-4 | trained | 68.754 | 30.595 | 27 | 31.767 |
| Yoochoose 1/64 | results-5 | trained | 67.632 | 30.137 | 24 | 31.961 |
| Yoochoose 1/64 | results-6 | trained | 68.714 | 30.558 | 27 | 31.819 |
| Yoochoose 1/64 | results-7 | trained | 68.489 | 30.516 | 22 | 31.960 |
| Yoochoose 1/64 | results-8 | trained | 70.183 | 31.057 | 8 | 32.679 |
| Yoochoose 1/64 | results-9 | trained | 70.174 | 31.172 | 11 | 32.733 |


![GAT-SR-GNN metric evolution](results-analysis/gat_sr_gnn_evolution.png)


## Model Evolution: GAT-TAGNN

`GAT-TAGNN` has trained exports for iterations 1, 3, 5, 6, 7, 8, and 9. Iteration 9 keeps the weighted `GATConv(edge_dim=1)` encoder from iteration 8, restores dropout `0.10`, and adds label smoothing `0.01`. It sets the best Yoochoose direct TAGNN-style GAT result and the best Diginetica Precision@20, but iteration 8 still has slightly better Diginetica MRR@20.


| Dataset | Run | Status | Precision@20 | MRR@20 | Best Epoch | Best Val MRR@20 |
| --- | --- | --- | --- | --- | --- | --- |
| Diginetica | results-1 | trained | 51.252 | 17.353 | 4 | 19.651 |
| Diginetica | results-2 | pending |  |  |  |  |
| Diginetica | results-3 | trained | 50.925 | 17.128 | 25 | 19.505 |
| Diginetica | results-4 | pending |  |  |  |  |
| Diginetica | results-5 | trained | 47.660 | 15.651 | 7 | 18.235 |
| Diginetica | results-6 | trained | 49.553 | 16.823 | 26 | 19.367 |
| Diginetica | results-7 | trained | 48.909 | 16.330 | 23 | 19.099 |
| Diginetica | results-8 | trained | 51.313 | 17.374 | 4 | 19.623 |
| Diginetica | results-9 | trained | 51.351 | 17.345 | 4 | 19.670 |
| Yoochoose 1/64 | results-1 | trained | 70.611 | 31.004 | 10 | 32.669 |
| Yoochoose 1/64 | results-2 | pending |  |  |  |  |
| Yoochoose 1/64 | results-3 | trained | 68.591 | 29.892 | 30 | 31.336 |
| Yoochoose 1/64 | results-4 | pending |  |  |  |  |
| Yoochoose 1/64 | results-5 | trained | 66.890 | 29.643 | 21 | 31.492 |
| Yoochoose 1/64 | results-6 | trained | 67.174 | 29.765 | 30 | 31.131 |
| Yoochoose 1/64 | results-7 | trained | 67.691 | 29.630 | 29 | 31.405 |
| Yoochoose 1/64 | results-8 | trained | 70.630 | 30.965 | 9 | 32.716 |
| Yoochoose 1/64 | results-9 | trained | 70.645 | 31.038 | 11 | 32.686 |


![GAT-TAGNN metric evolution](results-analysis/gat_tagnn_evolution.png)


## Third Architecture: GAT-SAGPool

`GAT-SAGPool` is a separate readout architecture rather than an iteration of the direct SR-GNN/TAGNN readout branches. The current workspace snapshot does not include `results-gat-sagpool/`, so the rebuild code skips SAGPool-specific tables and plots until that folder is restored.


SAGPool plots are skipped in this snapshot because `results-gat-sagpool/` is not available.


## Validation Curves

The validation curves are supporting diagnostics for the exported checkpoints. The main comparison remains the held-out test `Precision@20` and `MRR@20` above.


![GAT-SR-GNN validation history](results-analysis/gat_sr_gnn_validation_history.png)

![GAT-TAGNN validation history](results-analysis/gat_tagnn_validation_history.png)

SAGPool validation history is skipped in this snapshot because `results-gat-sagpool/` is not available.


## Reproducibility: Build Tables and Plots

The cells below rebuild every table and figure used above directly from the archived result folders. Running them updates the derived files under `results-analysis/`, including the main best-vs-baseline plots, the direct architecture comparison, the per-model evolution plots, and the SAGPool plots.


In [1]:
from pathlib import Path
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
PRESENTATION_DIR = PROJECT_ROOT / "results-analysis"
PRESENTATION_DIR.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:.3f}".format

BASELINE_ROWS = [
    ("POP", 6.71, 1.65, 0.89, 0.20),
    ("S-POP", 30.44, 18.35, 21.06, 13.68),
    ("Item-KNN", 51.60, 21.81, 35.75, 11.57),
    ("BPR-MF", 31.31, 12.08, 5.24, 1.98),
    ("FPMC", 45.62, 15.01, 26.53, 6.95),
    ("GRU4REC", 60.64, 22.89, 29.45, 8.33),
    ("NARM", 68.32, 28.63, 49.70, 16.17),
    ("STAMP", 68.74, 29.67, 45.64, 14.32),
    ("SR-GNN", 70.57, 30.94, 50.73, 17.59),
    ("TAGNN", 71.02, 31.12, 51.31, 18.03),
]
BASELINE_COLUMNS = [
    "method", "yoochoose_1_64_precision@20", "yoochoose_1_64_mrr@20",
    "diginetica_precision@20", "diginetica_mrr@20",
]
baselines = pd.DataFrame(BASELINE_ROWS, columns=BASELINE_COLUMNS)

ITERATIONS = [
    {"iteration": 1, "folder": "results-1", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "Initial complete GAT-SR-GNN and GAT-TAGNN archive used as the first working reference point."},
    {"iteration": 2, "folder": "results-2", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "pending", "What Changed": "Simplified GAT stack revision; only GAT-SR-GNN was rerun in this export."},
    {"iteration": 3, "folder": "results-3", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "GATv2/performance iteration, including TAGNN scoring/chunking changes and SR-GNN positional-embedding rollback."},
    {"iteration": 4, "folder": "results-4", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "pending", "What Changed": "Repeated transition edges plus explicit residual connections in the GAT stack; TAGNN still pending."},
    {"iteration": 5, "folder": "results-5", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "Failed edge-attribute/gated-normalization ablation: normalized edge_attr for GATv2Conv(edge_dim=1) with gated residual blending and LayerNorm per directional GAT stack; hidden_dim remains 100."},
    {"iteration": 6, "folder": "results-6", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "Shared GAT-only recovery iteration with repeated transition edges by default, no scalar edge_attr, GATv2Conv, plain residual and one layer; SR-GNN/TAGNN readouts unchanged."},
    {"iteration": 7, "folder": "results-7", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "GAT-only identity-preserving fusion: same iteration-6 GATv2 stack, then add original item embedding after direction fusion; SR-GNN/TAGNN readouts unchanged."},
    {"iteration": 8, "folder": "results-8", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "Restored iteration-1 weighted GATConv(edge_dim=1) architecture from commit 7bff12d with mild dropout 0.15; both direct branches trained in results-8."},
    {"iteration": 9, "folder": "results-9", "GAT-SR-GNN status": "trained", "GAT-TAGNN status": "trained", "What Changed": "Same weighted GATConv(edge_dim=1) architecture as iteration 8, but uses dropout 0.10 plus small label smoothing 0.01 via r5_restored_baseline_label_smoothing; both direct branches trained in results-9."},
]
iteration_status = pd.DataFrame(ITERATIONS)
iteration_status.to_csv(PRESENTATION_DIR / "iteration_status.csv", index=False)

REFERENCE_BY_MODEL = {"GAT-SR-GNN": "SR-GNN", "GAT-TAGNN": "TAGNN", "GAT-SAGPool": "SR-GNN"}
COMPARISON_NOTE = {
    "GAT-SR-GNN": "direct SR-GNN readout comparison",
    "GAT-TAGNN": "direct TAGNN readout comparison",
    "GAT-SAGPool": "contextual graph-session baseline; SAGPool is a different readout",
}
DATASET_KEY = {"Yoochoose 1/64": "yoochoose_1_64", "Diginetica": "diginetica"}
EXPECTED_RESULT_COLUMNS = [
    "model", "iteration", "folder", "dataset", "test_precision@20", "test_mrr@20", "test_loss",
    "best_epoch", "best_validation_precision@20", "best_validation_mrr@20", "train_examples",
    "validation_examples", "test_examples", "num_items", "checkpoint_path",
]
iteration_status


,iteration,folder,GAT-SR-GNN status,GAT-TAGNN status,What Changed
0,1,results-1,trained,trained,Initial complete GAT-SR-GNN and GAT-TAGNN arch...
1,2,results-2,trained,pending,Simplified GAT stack revision; only GAT-SR-GNN...
2,3,results-3,trained,trained,"GATv2/performance iteration, including TAGNN s..."
3,4,results-4,trained,pending,Repeated transition edges plus explicit residu...
4,5,results-5,trained,trained,Failed edge-attribute/gated-normalization abla...
5,6,results-6,trained,trained,Shared GAT-only recovery iteration with repeat...
6,7,results-7,trained,trained,GAT-only identity-preserving fusion: same iter...
7,8,results-8,trained,trained,Restored iteration-1 weighted GATConv(edge_dim...
8,9,results-9,trained,trained,Same weighted GATConv(edge_dim=1) architecture...


In [2]:
def load_result_file(path, model, iteration, folder):
    if not path.is_file():
        return None
    df = pd.read_csv(path).assign(model=model, iteration=iteration, folder=folder)
    for column in EXPECTED_RESULT_COLUMNS:
        if column not in df.columns:
            df[column] = pd.NA
    return df[EXPECTED_RESULT_COLUMNS]

run_frames = []
for run in ITERATIONS:
    folder = PROJECT_ROOT / run["folder"]
    for filename, model in [
        ("gat_sr_gnn_results.csv", "GAT-SR-GNN"),
        ("gat_tagnn_results.csv", "GAT-TAGNN"),
    ]:
        loaded = load_result_file(folder / filename, model, run["iteration"], run["folder"])
        if loaded is not None:
            run_frames.append(loaded)

sagpool_results = load_result_file(
    PROJECT_ROOT / "results-gat-sagpool" / "gat_sagpool_results.csv",
    "GAT-SAGPool",
    1,
    "results-gat-sagpool",
)
if sagpool_results is not None:
    run_frames.append(sagpool_results)

all_runs = (
    pd.concat(run_frames, ignore_index=True)
    .sort_values(["model", "dataset", "iteration"])
    .reset_index(drop=True)
)
all_runs.to_csv(PRESENTATION_DIR / "all_model_runs.csv", index=False)
all_runs


,model,iteration,folder,dataset,test_precision@20,test_mrr@20,test_loss,best_epoch,best_validation_precision@20,best_validation_mrr@20,train_examples,validation_examples,test_examples,num_items,checkpoint_path
0,GAT-SR-GNN,1,results-1,Diginetica,50.250,16.646,5.553,4,55.795,19.413,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
1,GAT-SR-GNN,2,results-2,Diginetica,50.629,16.725,5.657,28,55.207,19.486,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
2,GAT-SR-GNN,3,results-3,Diginetica,50.736,16.855,5.654,19,55.478,19.563,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
3,GAT-SR-GNN,4,results-4,Diginetica,50.454,16.802,5.664,19,55.243,19.529,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
4,GAT-SR-GNN,5,results-5,Diginetica,48.173,15.789,6.011,10,53.293,18.889,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
5,GAT-SR-GNN,6,results-6,Diginetica,50.391,16.759,5.671,28,55.227,19.532,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
6,GAT-SR-GNN,7,results-7,Diginetica,50.266,16.696,5.684,26,55.150,19.471,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
7,GAT-SR-GNN,8,results-8,Diginetica,50.299,16.569,5.553,4,55.804,19.462,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
8,GAT-SR-GNN,9,results-9,Diginetica,50.332,16.664,5.641,4,55.849,19.447,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...
9,GAT-SR-GNN,1,results-1,Yoochoose 1/64,70.124,31.146,4.342,9,69.474,32.706,332874,36985,55898,37484,/kaggle/working/results/checkpoints/gat_sr_gnn...


In [3]:
best_rows = []
for (model, dataset), group in all_runs.groupby(["model", "dataset"], sort=False):
    best = group.sort_values(
        ["test_mrr@20", "test_precision@20", "iteration"],
        ascending=[False, False, False],
    ).iloc[0]
    reference_method = REFERENCE_BY_MODEL[model]
    dataset_key = DATASET_KEY[dataset]
    reference_precision = baselines.loc[
        baselines["method"] == reference_method,
        f"{dataset_key}_precision@20",
    ].iloc[0]
    reference_mrr = baselines.loc[
        baselines["method"] == reference_method,
        f"{dataset_key}_mrr@20",
    ].iloc[0]
    best_rows.append({
        "model": model,
        "dataset": dataset,
        "best_iteration": int(best["iteration"]),
        "folder": best["folder"],
        "reference_method": reference_method,
        "comparison_note": COMPARISON_NOTE[model],
        "test_precision@20": best["test_precision@20"],
        "reference_precision@20": reference_precision,
        "delta_precision@20": best["test_precision@20"] - reference_precision,
        "test_mrr@20": best["test_mrr@20"],
        "reference_mrr@20": reference_mrr,
        "delta_mrr@20": best["test_mrr@20"] - reference_mrr,
        "best_epoch": best["best_epoch"],
        "best_validation_mrr@20": best["best_validation_mrr@20"],
    })

best_vs_baseline = (
    pd.DataFrame(best_rows)
    .sort_values(["model", "dataset"])
    .reset_index(drop=True)
)
best_vs_baseline.to_csv(PRESENTATION_DIR / "best_vs_baseline.csv", index=False)

gat_sagpool_architecture_results = all_runs[all_runs["model"] == "GAT-SAGPool"].copy()
gat_sagpool_architecture_results.to_csv(PRESENTATION_DIR / "gat_sagpool_architecture_results.csv", index=False)
best_vs_baseline


,model,dataset,best_iteration,folder,reference_method,comparison_note,test_precision@20,reference_precision@20,delta_precision@20,test_mrr@20,reference_mrr@20,delta_mrr@20,best_epoch,best_validation_mrr@20
0,GAT-SR-GNN,Diginetica,3,results-3,SR-GNN,direct SR-GNN readout comparison,50.736,50.730,0.006,16.855,17.590,-0.735,19,19.563
1,GAT-SR-GNN,Yoochoose 1/64,9,results-9,SR-GNN,direct SR-GNN readout comparison,70.174,70.570,-0.396,31.172,30.940,0.232,11,32.733
2,GAT-TAGNN,Diginetica,8,results-8,TAGNN,direct TAGNN readout comparison,51.313,51.310,0.003,17.374,18.030,-0.656,4,19.623
3,GAT-TAGNN,Yoochoose 1/64,9,results-9,TAGNN,direct TAGNN readout comparison,70.645,71.020,-0.375,31.038,31.120,-0.082,11,32.686


In [4]:
plot_best = best_vs_baseline.copy()
plot_best["label"] = (
    plot_best["model"].str.replace("GAT-", "", regex=False)
    + "\n"
    + plot_best["dataset"]
)
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.2))
for ax, metric, reference_metric, ylabel, title in [
    (axes[0], "test_precision@20", "reference_precision@20", "Precision@20", "Best achieved Precision@20 vs paper/context baseline"),
    (axes[1], "test_mrr@20", "reference_mrr@20", "MRR@20", "Best achieved MRR@20 vs paper/context baseline"),
]:
    x_positions = list(range(len(plot_best)))
    width = 0.38
    ax.bar([x - width / 2 for x in x_positions], plot_best[reference_metric], width=width, label="Published baseline", color="#8c8c8c")
    bars = ax.bar([x + width / 2 for x in x_positions], plot_best[metric], width=width, label="GAT architecture", color="#4c72b0")
    ax.set_xticks(x_positions, plot_best["label"])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.margins(y=0.18)
fig.tight_layout()
fig.savefig(PRESENTATION_DIR / "best_vs_baseline.png", dpi=180, bbox_inches="tight")
plt.show()

# Delta from the relevant baseline for each architecture/dataset pair.
delta_plot = best_vs_baseline.melt(
    id_vars=["model", "dataset", "best_iteration"],
    value_vars=["delta_precision@20", "delta_mrr@20"],
    var_name="metric",
    value_name="delta",
)
delta_plot["metric"] = delta_plot["metric"].map({"delta_precision@20": "Precision@20", "delta_mrr@20": "MRR@20"})
delta_plot["label"] = (
    delta_plot["model"].str.replace("GAT-", "", regex=False)
    + "\n"
    + delta_plot["dataset"]
    + "\n"
    + delta_plot["metric"]
)
fig, ax = plt.subplots(figsize=(13.5, 5.2))
bars = ax.bar(
    delta_plot["label"],
    delta_plot["delta"],
    color=["#3a923a" if value >= 0 else "#c44e52" for value in delta_plot["delta"]],
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Absolute delta from published/context baseline")
ax.set_title("Best architecture result delta vs matching baseline")
ax.tick_params(axis="x", rotation=45)
ax.bar_label(bars, fmt="%+.2f", padding=3, fontsize=8)
ax.margins(y=0.25)
fig.tight_layout()
fig.savefig(PRESENTATION_DIR / "best_delta_vs_baseline.png", dpi=180, bbox_inches="tight")
plt.show()


/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/2691046677.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/2691046677.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
available_models = [model for model in ["GAT-SR-GNN", "GAT-TAGNN", "GAT-SAGPool"] if model in best_vs_baseline["model"].unique()]
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
for ax, metric, ylabel in [
    (axes[0], "test_precision@20", "Precision@20"),
    (axes[1], "test_mrr@20", "MRR@20"),
]:
    pivot = best_vs_baseline.pivot(index="dataset", columns="model", values=metric)[available_models]
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"Best GAT architecture {ylabel}")
    ax.set_xlabel("Dataset")
    ax.set_ylabel(ylabel)
    ax.legend(title="Architecture")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3, fontsize=8)
    ax.margins(y=0.2)
fig.tight_layout()
fig.savefig(PRESENTATION_DIR / "architecture_best_comparison.png", dpi=180, bbox_inches="tight")
plt.show()


/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/2596112332.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
evolution_rows = []
for model in ["GAT-SR-GNN", "GAT-TAGNN"]:
    datasets = sorted(all_runs.loc[all_runs["model"] == model, "dataset"].unique())
    for dataset in datasets:
        existing = all_runs[(all_runs["model"] == model) & (all_runs["dataset"] == dataset)].set_index("iteration")
        for run in ITERATIONS:
            iteration = run["iteration"]
            if iteration in existing.index:
                record = existing.loc[iteration].to_dict()
                record.update({"model": model, "dataset": dataset, "iteration": int(iteration), "folder": run["folder"], "status": "trained"})
            else:
                record = {column: pd.NA for column in EXPECTED_RESULT_COLUMNS}
                record.update({"model": model, "dataset": dataset, "iteration": int(iteration), "folder": run["folder"], "checkpoint_path": "pending rerun", "status": "pending"})
            evolution_rows.append(record)

model_evolution = (
    pd.DataFrame(evolution_rows)
    .sort_values(["model", "dataset", "iteration"])
    .reset_index(drop=True)
)
model_evolution.to_csv(PRESENTATION_DIR / "model_evolution.csv", index=False)
model_evolution


,model,folder,dataset,test_precision@20,test_mrr@20,test_loss,best_epoch,best_validation_precision@20,best_validation_mrr@20,train_examples,validation_examples,test_examples,num_items,checkpoint_path,iteration,status
0,GAT-SR-GNN,results-1,Diginetica,50.250,16.646,5.553,4,55.795,19.413,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,1,trained
1,GAT-SR-GNN,results-2,Diginetica,50.629,16.725,5.657,28,55.207,19.486,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,2,trained
2,GAT-SR-GNN,results-3,Diginetica,50.736,16.855,5.654,19,55.478,19.563,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,3,trained
3,GAT-SR-GNN,results-4,Diginetica,50.454,16.802,5.664,19,55.243,19.529,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,4,trained
4,GAT-SR-GNN,results-5,Diginetica,48.173,15.789,6.011,10,53.293,18.889,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,5,trained
5,GAT-SR-GNN,results-6,Diginetica,50.391,16.759,5.671,28,55.227,19.532,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,6,trained
6,GAT-SR-GNN,results-7,Diginetica,50.266,16.696,5.684,26,55.150,19.471,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,7,trained
7,GAT-SR-GNN,results-8,Diginetica,50.299,16.569,5.553,4,55.804,19.462,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,8,trained
8,GAT-SR-GNN,results-9,Diginetica,50.332,16.664,5.641,4,55.849,19.447,647523,71947,60858,43098,/kaggle/working/results/checkpoints/gat_sr_gnn...,9,trained
9,GAT-SR-GNN,results-1,Yoochoose 1/64,70.124,31.146,4.342,9,69.474,32.706,332874,36985,55898,37484,/kaggle/working/results/checkpoints/gat_sr_gnn...,1,trained


In [7]:
def plot_evolution(model, filename):
    model_runs = model_evolution[(model_evolution["model"] == model) & (model_evolution["status"] == "trained")].copy()
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharex=True)
    for ax, metric, ylabel in [(axes[0], "test_precision@20", "Precision@20"), (axes[1], "test_mrr@20", "MRR@20")]:
        for dataset, dataset_runs in model_runs.groupby("dataset"):
            dataset_runs = dataset_runs.sort_values("iteration")
            ax.plot(dataset_runs["iteration"], dataset_runs[metric], marker="o", linewidth=2, label=dataset)
            for _, row in dataset_runs.iterrows():
                ax.annotate(f"{row[metric]:.2f}", (row["iteration"], row[metric]), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=8)
        ax.set_xticks([run["iteration"] for run in ITERATIONS])
        ax.set_xlabel("Result iteration")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{model}: {ylabel} evolution")
        ax.legend(title="Dataset")
        ax.margins(y=0.2)
    fig.tight_layout()
    fig.savefig(PRESENTATION_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()

plot_evolution("GAT-SR-GNN", "gat_sr_gnn_evolution.png")
plot_evolution("GAT-TAGNN", "gat_tagnn_evolution.png")


/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/606162332.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/606162332.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
if "GAT-SAGPool" in best_vs_baseline["model"].unique():
    sagpool_context = best_vs_baseline[best_vs_baseline["model"] == "GAT-SAGPool"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
    for ax, metric, reference_metric, ylabel in [
        (axes[0], "test_precision@20", "reference_precision@20", "Precision@20"),
        (axes[1], "test_mrr@20", "reference_mrr@20", "MRR@20"),
    ]:
        x_positions = list(range(len(sagpool_context)))
        width = 0.38
        ax.bar([x - width / 2 for x in x_positions], sagpool_context[reference_metric], width=width, label="SR-GNN paper row", color="#8c8c8c")
        bars = ax.bar([x + width / 2 for x in x_positions], sagpool_context[metric], width=width, label="GAT-SAGPool", color="#55a868")
        ax.set_xticks(x_positions, sagpool_context["dataset"])
        ax.set_title(f"GAT-SAGPool {ylabel} with SR-GNN context")
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
        ax.margins(y=0.18)
    fig.tight_layout()
    fig.savefig(PRESENTATION_DIR / "gat_sagpool_baseline_context.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Skipping SAGPool context plot: results-gat-sagpool/ is not available.")


Skipping SAGPool context plot: results-gat-sagpool/ is not available.


In [9]:
history_frames = []
for run in ITERATIONS:
    for filename, model in [("gat_sr_gnn_history.csv", "GAT-SR-GNN"), ("gat_tagnn_history.csv", "GAT-TAGNN")]:
        path = PROJECT_ROOT / run["folder"] / filename
        if path.is_file():
            history_frames.append(pd.read_csv(path).assign(model=model, iteration=int(run["iteration"]), folder=run["folder"]))

sagpool_history_path = PROJECT_ROOT / "results-gat-sagpool" / "gat_sagpool_history.csv"
if sagpool_history_path.is_file():
    history_frames.append(pd.read_csv(sagpool_history_path).assign(model="GAT-SAGPool", iteration=1, folder="results-gat-sagpool"))

training_histories = pd.concat(history_frames, ignore_index=True)
training_histories.to_csv(PRESENTATION_DIR / "training_histories.csv", index=False)
training_histories.head()


,dataset,epoch,train_loss,validation_loss,validation_precision@20,validation_mrr@20,epoch_seconds,train_examples,validation_examples,test_examples,model,iteration,folder,learning_rate
0,Yoochoose 1/64,1,5.527,4.725,64.664,28.646,153.680,332874,36985,55898,GAT-SR-GNN,1,results-1,NaN
1,Yoochoose 1/64,2,4.456,4.514,67.460,30.068,148.083,332874,36985,55898,GAT-SR-GNN,1,results-1,NaN
2,Yoochoose 1/64,3,4.202,4.443,68.468,30.689,147.950,332874,36985,55898,GAT-SR-GNN,1,results-1,NaN
3,Yoochoose 1/64,4,3.773,4.346,69.636,32.411,149.080,332874,36985,55898,GAT-SR-GNN,1,results-1,NaN
4,Yoochoose 1/64,5,3.680,4.354,69.580,32.568,148.789,332874,36985,55898,GAT-SR-GNN,1,results-1,NaN


In [10]:
def plot_validation_history(model, filename):
    model_history = training_histories[training_histories["model"] == model].copy()
    datasets = list(model_history["dataset"].drop_duplicates())
    if not datasets:
        print(f"Skipping {model} validation history: no history rows available.")
        return
    fig, axes = plt.subplots(1, len(datasets), figsize=(6.5 * len(datasets), 4.6), squeeze=False)
    for ax, dataset in zip(axes[0], datasets):
        dataset_history = model_history[model_history["dataset"] == dataset]
        for iteration, run_history in dataset_history.groupby("iteration"):
            run_history = run_history.sort_values("epoch")
            label = f"results-{iteration}" if model != "GAT-SAGPool" else "results-gat-sagpool"
            ax.plot(run_history["epoch"], run_history["validation_mrr@20"], label=label, linewidth=1.8)
        ax.set_title(f"{model}: {dataset} validation MRR@20")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation MRR@20")
        ax.legend(title="Run")
        ax.margins(y=0.15)
    fig.tight_layout()
    fig.savefig(PRESENTATION_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()

plot_validation_history("GAT-SR-GNN", "gat_sr_gnn_validation_history.png")
plot_validation_history("GAT-TAGNN", "gat_tagnn_validation_history.png")
plot_validation_history("GAT-SAGPool", "gat_sagpool_validation_history.png")


/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/3664196444.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Skipping GAT-SAGPool validation history: no history rows available.


/var/folders/z8/r1hd_2mj23q_bxkpbnv1fc6c0000gn/T/ipykernel_42016/3664196444.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Dedicated SAGPool comparison plots. These run only when results-gat-sagpool/ is available.
if "GAT-SAGPool" in best_vs_baseline["model"].unique():
    sagpool_comparison = best_vs_baseline.copy()
    model_order = [model for model in ["GAT-SR-GNN", "GAT-TAGNN", "GAT-SAGPool"] if model in sagpool_comparison["model"].unique()]

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))
    for ax, metric, ylabel in [
        (axes[0], "test_precision@20", "Precision@20"),
        (axes[1], "test_mrr@20", "MRR@20"),
    ]:
        pivot = sagpool_comparison.pivot(index="dataset", columns="model", values=metric)[model_order]
        pivot.plot(kind="bar", ax=ax, rot=0)
        ax.set_title(f"GAT-SAGPool vs other GAT architectures: {ylabel}")
        ax.set_xlabel("Dataset")
        ax.set_ylabel(ylabel)
        ax.legend(title="Architecture")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.2f", padding=3, fontsize=8)
        ax.margins(y=0.2)
    fig.tight_layout()
    fig.savefig(PRESENTATION_DIR / "gat_sagpool_vs_architectures.png", dpi=180, bbox_inches="tight")
    plt.show()

    sagpool_deltas = []
    for dataset in sorted(sagpool_comparison["dataset"].unique()):
        dataset_rows = sagpool_comparison[sagpool_comparison["dataset"] == dataset].set_index("model")
        sagpool_row = dataset_rows.loc["GAT-SAGPool"]
        for reference_model in [model for model in ["GAT-SR-GNN", "GAT-TAGNN"] if model in dataset_rows.index]:
            reference_row = dataset_rows.loc[reference_model]
            for metric, label in [("test_precision@20", "Precision@20"), ("test_mrr@20", "MRR@20")]:
                sagpool_deltas.append({
                    "dataset": dataset,
                    "comparison": f"SAGPool - {reference_model.replace('GAT-', '')}",
                    "metric": label,
                    "delta": sagpool_row[metric] - reference_row[metric],
                })

    sagpool_delta_df = pd.DataFrame(sagpool_deltas)
    sagpool_delta_df.to_csv(PRESENTATION_DIR / "gat_sagpool_delta_vs_architectures.csv", index=False)
    sagpool_delta_df["label"] = (
        sagpool_delta_df["dataset"]
        + "\n"
        + sagpool_delta_df["comparison"]
        + "\n"
        + sagpool_delta_df["metric"]
    )
    fig, ax = plt.subplots(figsize=(13.5, 5.0))
    bars = ax.bar(
        sagpool_delta_df["label"],
        sagpool_delta_df["delta"],
        color=["#3a923a" if value >= 0 else "#c44e52" for value in sagpool_delta_df["delta"]],
    )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title("GAT-SAGPool delta vs other GAT architectures")
    ax.set_ylabel("SAGPool metric minus comparison model")
    ax.tick_params(axis="x", rotation=45)
    ax.bar_label(bars, fmt="%+.2f", padding=3, fontsize=8)
    ax.margins(y=0.25)
    fig.tight_layout()
    fig.savefig(PRESENTATION_DIR / "gat_sagpool_delta_vs_architectures.png", dpi=180, bbox_inches="tight")
    plt.show()

    validation_comparison = best_vs_baseline[[
        "model",
        "dataset",
        "best_validation_mrr@20",
    ]].copy()
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    validation_pivot = validation_comparison.pivot(
        index="dataset",
        columns="model",
        values="best_validation_mrr@20",
    )[model_order]
    validation_pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title("Best validation MRR@20 across GAT architectures")
    ax.set_xlabel("Dataset")
    ax.set_ylabel("Best validation MRR@20")
    ax.legend(title="Architecture")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3, fontsize=8)
    ax.margins(y=0.2)
    fig.tight_layout()
    fig.savefig(PRESENTATION_DIR / "gat_sagpool_validation_vs_architectures.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("Skipping SAGPool comparison plots: results-gat-sagpool/ is not available.")


Skipping SAGPool comparison plots: results-gat-sagpool/ is not available.
